# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [13]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [14]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [15]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [16]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [17]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction()

In [18]:
try:
    # Attempt to create the collection
    collection = chroma_client.create_collection(
        name="udaplay",
        embedding_function=embedding_fn
    )
    print("Collection 'udaplay' created. Adding games to database...")
    
    data_dir = "games"
    for file_name in sorted(os.listdir(data_dir)):
        if not file_name.endswith(".json"):
            continue
        
        file_path = os.path.join(data_dir, file_name)
        with open(file_path, "r", encoding="utf-8") as f_game:
            game = json.load(f_game)
            
        content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
        doc_id = os.path.splitext(file_name)[0]
        
        collection.add(
            ids=[doc_id],
            documents=[content],
            metadatas=[game]
        )
    print("Successfully added all games to the database.")
except Exception as e:
    # If it fails (e.g. collection already exists), get the collection instead
    print(f"Collection may already exist. Attempting to retrieve it. Error: {e}")
    collection = chroma_client.get_collection(
        name="udaplay",
        embedding_function=embedding_fn
    )
    print("Successfully retrieved existing collection 'udaplay'.")

Collection may already exist. Attempting to retrieve it. Error: Collection [udaplay] already exists
Successfully retrieved existing collection 'udaplay'.


### Add documents

In [19]:
# Games are now added inside the try-catch block when creating the collection.
print(f"Total documents in collection: {collection.count()}")

Total documents in collection: 15


In [ ]:
# Search demo
query = "Adventure games"
results = collection.query(query_texts=[query], n_results=3)
for meta, dist in zip(results["metadatas"][0], results["distances"][0]):
    print(f"{meta['Name']} ({meta['Platform']})  distance={dist:.4f}")

Minecraft (Xbox One)  distance=0.1796
Kinect Adventures! (Xbox 360)  distance=0.1921
Grand Theft Auto: San Andreas (PlayStation 2)  distance=0.1982
